# 🏥 NHS Claims — Data Engineering & Semantic Pipeline
## Bronze → Silver → Gold | Microsoft Fabric Medallion Architecture

> **Note:** Portfolio demonstration project. All data is mocked and fictitious.  
> Inspired by real NHS-adjacent data engineering work using Microsoft Fabric and Apache Spark.

### What this notebook covers:
1. Run the full Bronze → Silver → Gold pipeline
2. Explore dimension and fact tables
3. Compute KPIs and reporting aggregations
4. Validate the semantic model outputs
5. Produce Power BI-ready reporting summaries
6. Visualise key metrics

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys, os

sys.path.insert(0, os.path.join('..', 'src'))
from pipeline import run_pipeline, build_gold_trust_kpis, build_gold_monthly_trends

pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Setup complete ✅')

## 1. Run the Pipeline

In [ ]:
# Run the full Bronze → Silver → Gold pipeline
# In Microsoft Fabric, each layer would be a separate Notebook
# triggered by a Data Factory pipeline or Fabric scheduler
tables = run_pipeline()

## 2. Explore the Silver Layer

In [ ]:
# Fact table
fact = tables['fact_claims']
print(f'fact_claims: {len(fact):,} rows × {len(fact.columns)} columns')
print(f'Columns: {list(fact.columns)}')
fact.head(3)

In [ ]:
# Date dimension — shows NHS financial year logic
dim_date = tables['dim_date']
print('Date dimension sample around FY boundary (March/April):')
dim_date[
    dim_date['calendar_date'].isin(
        pd.date_range('2023-03-29', '2023-04-03')
    )
][['calendar_date', 'calendar_month_name', 'nhs_financial_year', 'nhs_financial_quarter', 'is_weekday']]

In [ ]:
# Fact table null check — approved_amount should have no nulls after COALESCE
print('=== Fact Table Data Quality ===')
print(f'Nulls in approved_amount: {fact["approved_amount"].isna().sum()}')
print(f'Min approved_amount: {fact["approved_amount"].min():,.2f}')
print(f'Records with 0 approved (approved status): '
      f'{len(fact[(fact["approved_amount"] == 0) & (fact["is_approved_flag"] == 1)]):,}')

print('\nAge band distribution:')
print(fact['patient_age_band'].value_counts().sort_index())

print('\nLifecycle stage distribution:')
print(fact['claim_lifecycle_stage'].value_counts())

## 3. Gold Layer — KPI Tables

In [ ]:
# Trust KPIs — the primary Power BI semantic layer feed
kpis = tables['gold_trust_kpis']

print('=== TRUST KPIs (Gold Layer) ===')
display_cols = [
    'nhs_trust_name', 'financial_year', 'total_claims',
    'approval_rate_pct', 'total_approved_gbp', 'financial_approval_rate_pct',
    'avg_submission_lag'
]
print(kpis[display_cols].to_string(index=False))

In [ ]:
# Aggregated overall figures (mimics Power BI card visuals)
print('=== OVERALL KPIs (All Trusts, All Years) ===')

total_claims   = fact['claim_id'].nunique()
total_claimed  = fact['claimed_amount'].sum()
total_approved = fact['approved_amount'].sum()
approval_rate  = fact['is_approved_flag'].mean() * 100
avg_lag        = fact['submission_lag_days'].mean()
late_pct       = (fact['submission_lag_days'] > 14).mean() * 100

print(f'  Total Claims             : {total_claims:,}')
print(f'  Total Claimed (£)        : £{total_claimed:,.0f}')
print(f'  Total Approved (£)       : £{total_approved:,.0f}')
print(f'  Claim Approval Rate      : {approval_rate:.1f}%')
print(f'  Avg Submission Lag       : {avg_lag:.1f} days')
print(f'  Late Submission Rate     : {late_pct:.1f}%')
print(f'  Variance (Claimed-Approved): £{total_claimed - total_approved:,.0f}')

In [ ]:
# Procedure summary
proc = tables['gold_procedure_summary']
print('=== PROCEDURE SUMMARY ===')
print(proc[['procedure_code','procedure_desc','procedure_category',
            'total_claims','approval_rate_pct','total_approved_gbp']].to_string(index=False))

## 4. Semantic Model Validation

Before connecting Power BI, we validate the semantic layer outputs.

In [ ]:
# Check referential integrity: all FK values in fact should exist in dims
print('=== REFERENTIAL INTEGRITY CHECKS ===')

dim_trust_ids    = set(tables['dim_trust']['nhs_trust_id'])
dim_provider_ids = set(tables['dim_provider']['provider_id'])
dim_proc_codes   = set(tables['dim_procedure']['procedure_code'])
dim_date_keys    = set(tables['dim_date']['date_key'])

checks = [
    ('nhs_trust_id',       set(fact['nhs_trust_id']),          dim_trust_ids),
    ('provider_id',        set(fact['provider_id']),           dim_provider_ids),
    ('procedure_code',     set(fact['procedure_code']),        dim_proc_codes),
    ('service_date_key',   set(fact['service_date_key']),      dim_date_keys),
]

for col, fact_vals, dim_vals in checks:
    orphans = fact_vals - dim_vals
    status = '✅ PASS' if not orphans else f'❌ FAIL — {len(orphans)} orphan values'
    print(f'  {col}: {status}')

In [ ]:
# Cross-check: Gold layer totals should match Silver fact totals
print('=== GOLD vs SILVER AGGREGATE RECONCILIATION ===')

silver_total_claims   = len(fact)
silver_total_approved = fact['approved_amount'].sum()

gold_total_claims   = kpis['total_claims'].sum()
gold_total_approved = kpis['total_approved_gbp'].sum()

print(f'  Total Claims   — Silver: {silver_total_claims:,} | Gold: {gold_total_claims:,} | Match: {silver_total_claims == gold_total_claims}')
print(f'  Total Approved — Silver: £{silver_total_approved:,.2f} | Gold: £{gold_total_approved:,.2f} | '
      f'Δ = {abs(silver_total_approved - gold_total_approved):.2f}')

## 5. Visualisations (Power BI Preview)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('NHS Claims — Semantic Model KPI Preview', fontsize=14, fontweight='bold')

# --- Plot 1: Approval rate by trust (bar) ---
trust_summary = kpis.groupby('nhs_trust_name').agg(
    approval_rate=('approval_rate_pct', 'mean'),
    fin_approval=('financial_approval_rate_pct', 'mean'),
).reset_index()
trust_names_short = [n.replace(' NHS Trust','').replace(' Health Trust','').replace(' Network','') 
                     for n in trust_summary['nhs_trust_name']]

x = range(len(trust_summary))
axes[0,0].bar([i - 0.2 for i in x], trust_summary['approval_rate'], 0.35, label='Claim Approval %', color='#2196F3')
axes[0,0].bar([i + 0.2 for i in x], trust_summary['fin_approval'], 0.35, label='Financial Approval %', color='#4CAF50')
axes[0,0].set_xticks(list(x))
axes[0,0].set_xticklabels(trust_names_short, rotation=15, ha='right', fontsize=8)
axes[0,0].set_title('Approval Rates by Trust (Avg across FYs)')
axes[0,0].set_ylabel('Rate (%)')
axes[0,0].legend(fontsize=8)
axes[0,0].set_ylim(0, 70)

# --- Plot 2: Monthly claim volume trend ---
monthly = tables['gold_monthly_trends']
all_monthly = monthly.groupby('service_month').agg(claim_volume=('claim_volume','sum')).reset_index()
all_monthly['service_month_dt'] = pd.to_datetime(all_monthly['service_month'])
all_monthly = all_monthly.sort_values('service_month_dt')

axes[0,1].plot(range(len(all_monthly)), all_monthly['claim_volume'], marker='o', markersize=3,
               color='#9C27B0', linewidth=1.5)
axes[0,1].set_title('Monthly Claim Volume (All Trusts)')
axes[0,1].set_ylabel('Claims')
tick_positions = [i for i in range(0, len(all_monthly), 4)]
tick_labels = [all_monthly['service_month'].iloc[i][:7] for i in tick_positions]
axes[0,1].set_xticks(tick_positions)
axes[0,1].set_xticklabels(tick_labels, rotation=30, ha='right', fontsize=8)
axes[0,1].axvline(x=11, color='red', linestyle='--', alpha=0.4, label='FY boundary')
axes[0,1].axvline(x=23, color='red', linestyle='--', alpha=0.4)

# --- Plot 3: Procedure category breakdown ---
proc_cat = tables['gold_procedure_summary'].groupby('procedure_category').agg(
    total_approved=('total_approved_gbp','sum')
).reset_index().sort_values('total_approved', ascending=False)

axes[1,0].barh(proc_cat['procedure_category'], proc_cat['total_approved'] / 1e3,
               color=['#FF5722','#2196F3','#4CAF50','#FF9800','#9C27B0'])
axes[1,0].set_title('Total Approved (£K) by Procedure Category')
axes[1,0].set_xlabel('£ Thousands')

# --- Plot 4: Submission lag distribution ---
axes[1,1].hist(fact['submission_lag_days'].clip(0, 40), bins=30,
               color='#FF5722', edgecolor='white', alpha=0.8)
axes[1,1].axvline(x=14, color='red', linestyle='--', label='14-day threshold')
axes[1,1].set_title('Submission Lag Distribution (Days)')
axes[1,1].set_xlabel('Days')
axes[1,1].set_ylabel('Frequency')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('../reports/semantic_model_kpis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/semantic_model_kpis.png')

## 6. Oracle vs Spark SQL: Key Transformation Differences

| Step | Oracle SQL | Spark SQL | Notes |
|------|-----------|-----------|-------|
| Date series generation | `CONNECT BY LEVEL` | `SEQUENCE() + EXPLODE()` | Completely different paradigm |
| NULL fill | `NVL(x, 0)` | `COALESCE(x, 0)` | ANSI standard |
| Conditional | `DECODE(x, a, b)` | `CASE WHEN x=a THEN b` | DECODE not in Spark |
| Date format | `TO_CHAR(d, 'YYYY-MM')` | `DATE_FORMAT(d, 'yyyy-MM')` | Case-sensitive mask! |
| Date difference | `TRUNC(d1) - TRUNC(d2)` | `DATEDIFF(d1, d2)` | Simpler in Spark |
| String agg | `LISTAGG() WITHIN GROUP` | `ARRAY_JOIN(COLLECT_SET())` | |
| Row ID | `ROWNUM` | `ROW_NUMBER() OVER()` | Always need window in Spark |

## 7. Medallion Architecture Summary

```
BRONZE (Raw)
  nhs_claims.csv   → Loaded as-is, no transformation
  providers.csv    → Loaded as-is
  nhs_trusts.csv   → Loaded as-is

SILVER (Cleansed + Modelled)
  fact_claims      → Typed, NULLs resolved, derived fields added
  dim_date         → Full calendar + NHS financial year
  dim_trust        → Reference data, surrogate keys
  dim_provider     → Reference data, surrogate keys
  dim_procedure    → Clinical coding reference

GOLD (Reporting)
  trust_kpis       → Aggregated by trust + financial year
  monthly_trends   → Monthly volumes + cumulative YTD
  procedure_summary → Procedure-level performance
```

In **Microsoft Fabric**, each layer is a set of Delta tables in the **Lakehouse**.  
Power BI connects via the **SQL Analytics Endpoint** using Direct Lake mode — no data duplication.